# ☕ Simple RAG Demo — Java Edition
**Retrieval-Augmented Generation** implemented in pure Java — no Python libraries!

## Can Colab run Java?
> **Yes!** Google Colab runs on Linux VMs with Java pre-installed. We write `.java` source files,
> compile with `javac`, and run with `java` — all from notebook cells using shell commands (`!`).
> No kernel tricks, no extensions needed.

## What's different from the Python version?
The Python version uses `sentence-transformers` (neural embeddings) + FAISS (vector index).
These are Python-only libraries. In Java we implement **TF-IDF** similarity from scratch —
a classic, interpretable approach that shows exactly how vector search works under the hood.

### Steps:
1. 📚 Build a knowledge base (String array)
2. 📐 Embed documents as TF-IDF vectors (pure Java)
3. 🗂️ Build an in-memory vector index
4. 🔎 Query with cosine similarity
5. 💬 Build a RAG prompt from retrieved context

> 🦃 *Created by Crunch, managed at [Copilotclaw](https://github.com/Copilotclaw/trainingdemo)*

## ✅ Verify Java is available

In [ ]:
!java --version
!javac --version

## Step 1 — Write the Java Source
We write `RAGDemo.java` using a Python triple-quoted string and save it to disk.
The Java code implements the full RAG pipeline in ~120 lines.

In [ ]:
java_source = '''
import java.util.*;
import java.util.stream.*;

/**
 * Simple RAG Demo - Java Edition
 *
 * Implements Retrieval-Augmented Generation using:
 *   - TF-IDF vectorisation (no external libs)
 *   - Cosine similarity search
 *   - Prompt builder with retrieved context
 *
 * Compatible with Java 11+
 */
public class RAGDemo {

    // ── Step 1: Knowledge Base ────────────────────────────────────────────────
    // Same documents as the Python version
    static final String[] DOCUMENTS = {
        "RAG stands for Retrieval-Augmented Generation. It combines search with LLMs.",
        "FAISS is a library by Meta for efficient similarity search on dense vectors.",
        "Sentence transformers convert text into meaningful numerical embeddings.",
        "Local RAG runs entirely on your machine - no API calls, no data leaving your system.",
        "Ollama lets you run models like Llama 3 or Mistral locally with one command.",
        "Vector databases store embeddings and support fast approximate nearest-neighbor search.",
        "The retrieval step finds the top-k most relevant chunks from your knowledge base.",
        "Chunking splits long documents into smaller pieces before embedding them."
    };

    // ── Step 2: Tokenisation ──────────────────────────────────────────────────
    // Lowercase + strip punctuation + split on whitespace
    static String[] tokenize(String text) {
        return text.toLowerCase()
                   .replaceAll("[^a-z0-9 ]", " ")
                   .trim()
                   .split("\\s+");
    }

    // Build a shared vocabulary from all documents
    static Map<String, Integer> buildVocab(String[] docs) {
        Map<String, Integer> vocab = new LinkedHashMap<>();
        for (String doc : docs) {
            for (String w : tokenize(doc)) {
                vocab.putIfAbsent(w, vocab.size());
            }
        }
        return vocab;
    }

    // ── Step 3: TF-IDF Embedding ──────────────────────────────────────────────
    // TF  = how often a word appears in THIS document (normalised)
    // IDF = log( (total docs + 1) / (docs containing word + 1) ) + 1
    // TF-IDF = TF * IDF  (rewards rare-but-relevant words)
    static double[] embed(String text, Map<String, Integer> vocab, String[] corpus) {
        String[] tokens = tokenize(text);
        double[] vec = new double[vocab.size()];

        // Count term frequencies
        Map<String, Integer> tf = new HashMap<>();
        for (String t : tokens) {
            tf.merge(t, 1, Integer::sum);
        }

        // Compute TF-IDF for each vocabulary word
        for (Map.Entry<String, Integer> entry : vocab.entrySet()) {
            String word  = entry.getKey();
            int    idx   = entry.getValue();

            int termFreq = tf.getOrDefault(word, 0);
            if (termFreq == 0) continue;  // skip absent words

            double termTF = (double) termFreq / tokens.length;

            // Count documents that contain this word
            long df = Arrays.stream(corpus)
                            .filter(d -> Arrays.asList(tokenize(d)).contains(word))
                            .count();
            double idf = Math.log((double)(corpus.length + 1) / (df + 1)) + 1.0;

            vec[idx] = termTF * idf;
        }
        return vec;
    }

    // ── Step 4: Cosine Similarity & Search ───────────────────────────────────
    // Cosine similarity = dot(a,b) / (|a| * |b|)
    // Range: 0 (unrelated) to 1 (identical)
    static double cosine(double[] a, double[] b) {
        double dot = 0, na = 0, nb = 0;
        for (int i = 0; i < a.length; i++) {
            dot += a[i] * b[i];
            na  += a[i] * a[i];
            nb  += b[i] * b[i];
        }
        return (na == 0 || nb == 0) ? 0.0 : dot / (Math.sqrt(na) * Math.sqrt(nb));
    }

    // Find the top-k most similar documents for a query
    static List<int[]> search(String query, double[][] index,
                              Map<String, Integer> vocab, int k) {
        double[] qv = embed(query, vocab, DOCUMENTS);
        List<int[]> scores = new ArrayList<>();   // [docIndex, scoreAsIntBits]
        for (int i = 0; i < index.length; i++) {
            double sim = cosine(qv, index[i]);
            scores.add(new int[]{ i, Float.floatToIntBits((float) sim) });
        }
        // Sort descending by similarity
        scores.sort((a, b) -> Float.compare(
            Float.intBitsToFloat(b[1]), Float.intBitsToFloat(a[1])));
        return scores.subList(0, Math.min(k, scores.size()));
    }

    // ── Step 5: Build RAG Prompt ──────────────────────────────────────────────
    // In a real system you would pass this string to an LLM (Ollama, OpenAI, etc.)
    static String buildPrompt(String question, double[][] index,
                              Map<String, Integer> vocab) {
        StringBuilder ctx = new StringBuilder();
        for (int[] r : search(question, index, vocab, 3)) {
            ctx.append("- ").append(DOCUMENTS[r[0]]).append("\n");
        }
        return "You are a helpful assistant. Use the context below to answer the question.\n\n"
             + "Context:\n" + ctx
             + "\nQuestion: " + question
             + "\n\nAnswer:";
    }

    // ── Main ──────────────────────────────────────────────────────────────────
    public static void main(String[] args) {
        System.out.println("Simple RAG Demo - Java Edition");
        System.out.println("================================");

        // Build shared vocabulary from all documents
        Map<String, Integer> vocab = buildVocab(DOCUMENTS);
        System.out.printf("\n[1] Knowledge base: %d documents%n", DOCUMENTS.length);

        // Embed all documents into TF-IDF vectors (our "index")
        double[][] index = new double[DOCUMENTS.length][];
        for (int i = 0; i < DOCUMENTS.length; i++) {
            index[i] = embed(DOCUMENTS[i], vocab, DOCUMENTS);
        }
        System.out.printf("[2] Embeddings: TF-IDF, %d dimensions%n", vocab.size());
        System.out.printf("[3] Index built with %d vectors%n%n", index.length);

        // Query from command-line args or default
        String query = args.length > 0
            ? String.join(" ", args)
            : "How does local RAG work without internet?";

        System.out.printf("[4] Query: \"%s\"%n%n", query);

        System.out.println("Top 3 results:");
        for (int[] r : search(query, index, vocab, 3)) {
            double sim = cosine(embed(query, vocab, DOCUMENTS), index[r[0]]);
            System.out.printf("  [%.4f] %s%n", sim, DOCUMENTS[r[0]]);
        }

        System.out.println();
        System.out.println("--- RAG Prompt (send this to your LLM) ---");
        System.out.println(buildPrompt(query, index, vocab));
    }
}
'''

with open('RAGDemo.java', 'w') as f:
    f.write(java_source)

print('✅ RAGDemo.java written')

## Step 2 — Compile

In [ ]:
!javac RAGDemo.java && echo '✅ Compiled successfully'

## Step 3 — Run with the default query

In [ ]:
!java RAGDemo

## Step 4 — Try different queries
Pass a query as a command-line argument to see how retrieval changes.

In [ ]:
!java RAGDemo What is FAISS and why is it useful

In [ ]:
!java RAGDemo How do vector databases store embeddings

In [ ]:
# Try a query about something completely unrelated — what does TF-IDF return?
!java RAGDemo cooking recipes pasta carbonara

## 📐 How TF-IDF Works (vs Neural Embeddings)

| | TF-IDF (this notebook) | Neural Embeddings (Python version) |
|---|---|---|
| **Method** | Bag-of-words + term frequency | Deep learning (transformer) |
| **Dimension** | Vocabulary size (~50–100 here) | Fixed (e.g. 384 for MiniLM) |
| **Semantic understanding** | ❌ keyword matching only | ✅ understands synonyms, context |
| **Speed** | ⚡ near-instant | 🐢 model load + inference |
| **Dependencies** | ✅ none — pure Java | `sentence-transformers`, `faiss-cpu` |
| **Best for** | Learning, prototyping, exact terms | Production RAG, semantic search |

**TF** (Term Frequency) = how often a word appears in a document, normalised by document length.

**IDF** (Inverse Document Frequency) = `log( (N+1) / (df+1) ) + 1` — penalises common words like "the", "is".

**Cosine similarity** = `dot(a, b) / (|a| × |b|)` — measures angle between vectors (1.0 = identical, 0.0 = unrelated).

## 🏋️ Exercises

1. **Add more documents** — edit the `DOCUMENTS` array in `RAGDemo.java` and add 5 sentences about a topic you know. Re-compile and re-run.
2. **Change `top_k`** — modify the `3` in `search(query, index, vocab, 3)` to `1` or `5`. How does the prompt change?
3. **Unrelated query** — try `!java RAGDemo chocolate cake recipe` — why does FAISS still return results? What does this tell you about TF-IDF?
4. **Print the vocabulary** — add `System.out.println(vocab.keySet())` to `main()`. How many words are in it?
5. **Print a raw vector** — add `System.out.println(Arrays.toString(index[0]))` to see what a TF-IDF embedding looks like.

## 🚀 Next Steps

- **Replace TF-IDF with real embeddings** — use [Deep Java Library (DJL)](https://djl.ai/) with a HuggingFace model
- **Add a real LLM** — call Ollama's REST API from Java: `HttpClient` → `http://localhost:11434/api/generate`
- **Persist the index** — serialise the `double[][]` to disk with `ObjectOutputStream`
- **Scale up** — replace the brute-force cosine loop with [HNSW](https://github.com/nmslib/hnswlib) or [Vespa](https://vespa.ai/)

## 🐍 Compare with the Python version

Open [simple-rag-demo.ipynb](https://colab.research.google.com/github/Copilotclaw/trainingdemo/blob/main/simple-rag-demo.ipynb) side-by-side to compare:
- Python uses neural embeddings → better semantic match
- Java uses TF-IDF → easier to understand, no dependencies
- The RAG *pipeline* (embed → index → retrieve → prompt) is identical in both